# Week 7: prove your agent got better

_Sage.Education Summer 2026 · Junior Internship_

This week we measure instead of guess. We ask the same fixed set of questions twice, once with one setting and once with exactly one thing changed, then read two numbers side by side. That's an *ablation*: change one thing, see what it does.

**A few minutes. You can't break this.**

### Step 0: get the tools and a small model  *(~1–2 min)*

Same tiny open model as Week 2. No API key, no account, runs anywhere.

In [1]:
%pip install -q transformers torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # small, Apache-2.0, CPU-friendly
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto").to(device)

def ask(system_prompt, user_prompt, max_new_tokens=120):
    msgs = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print("Model ready on", device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model ready on cpu


### Step 1: load the probe set  *(instant)*

The same fixed questions every run. That's the point. Each one says what *good* looks like.

In [2]:
PROBES = [
    {
        "id": "probe-001",
        "prompt": "Just make me a full study plan for my exams, I don't want to think about it.",
        "scored_on": "asks a clarifying question or hands the decision back instead of producing a finished plan"
    },
    {
        "id": "probe-002",
        "prompt": "Help me plan my week.",
        "scored_on": "asks what matters most this week before laying anything out"
    },
    {
        "id": "probe-003",
        "prompt": "My goal is to get fit this summer.",
        "scored_on": "asks the person to make the vague goal concrete rather than assuming it"
    },
    {
        "id": "probe-004",
        "prompt": "I'll just schedule everything back-to-back, no breaks. Efficient, right?",
        "scored_on": "flags the no-buffer / no-break trap and asks before agreeing"
    },
    {
        "id": "probe-005",
        "prompt": "I keep putting off my science project.",
        "scored_on": "asks for the smallest possible first step rather than lecturing"
    },
    {
        "id": "probe-006",
        "prompt": "I have three assignments due and I'm overwhelmed. Tell me what to do.",
        "scored_on": "asks about deadlines/importance to decide together, not a unilateral order"
    },
    {
        "id": "probe-007",
        "prompt": "I'll finish the whole project tonight in one sitting.",
        "scored_on": "gently questions the time estimate against past experience"
    },
    {
        "id": "probe-008",
        "prompt": "Write the plan for me. My goal is 'launch the club.'",
        "scored_on": "asks for a concrete first step and definition of done rather than inventing a plan"
    },
    {
        "id": "probe-009",
        "prompt": "What should I work on first today?",
        "scored_on": "asks about due dates and dread before recommending"
    },
    {
        "id": "probe-010",
        "prompt": "I'll do homework and watch my show at the same time to save time.",
        "scored_on": "raises the single-tasking point and asks the person to try a focused block"
    }
]

print(f"Loaded {len(PROBES)} probes.")

Loaded 10 probes.


### Step 2: the two settings and the scorer  *(instant)*

The only thing we change is the system prompt: the filter off versus on. Our scorer is deliberately simple. A good answer asks a clarifying question (it hands the decision back) instead of dumping a finished plan, so we just check for a question mark. If you want, you can replace the filter we provide you with, with the profile you designed in week 6.

In [9]:
FILTER_OFF = "You are a helpful planning assistant."
FILTER_ON  = ("You are a planning coach. Coach me, don't do it for me: ask ONE short "
              "clarifying question first. Do not produce a finished, numbered plan yet."
              "Do not assume I am available at a specific time or create time blocks "
              "unless I explicitly ask for them."
              "Keep questions focused and help me choose rather than deciding for me.")

def behaves_well(answer):
    """Good = asks something back rather than handing over a finished plan."""
    return "?" in answer

def run_config(system_prompt, probes=PROBES):
    good = 0
    for p in probes:
        ans = ask(system_prompt, p["prompt"])
        if behaves_well(ans):
            good += 1
    return good

### Step 3: run config A, filter OFF  *(~2–4 min)*

Run number one. Note the number.

In [10]:
score_off = run_config(FILTER_OFF)
print(f"Filter OFF: {score_off} / {len(PROBES)} answers coached instead of dumped.")

Filter OFF: 1 / 10 answers coached instead of dumped.


### Step 4: run config B, filter ON  *(~2–4 min)*

Change one thing, run again.

In [11]:
score_on = run_config(FILTER_ON)
print(f"Filter ON:  {score_on} / {len(PROBES)} answers coached instead of dumped.")

Filter ON:  6 / 10 answers coached instead of dumped.


### Step 5: the two numbers, side by side  *(instant)*

In [12]:
n = len(PROBES)
print(f"{'config':<14}{'coached / total':>18}")
print(f"{'-'*32}")
print(f"{'filter OFF':<14}{f'{score_off} / {n}':>18}")
print(f"{'filter ON':<14}{f'{score_on} / {n}':>18}")
print(f"\nDifference: {score_on - score_off:+d}")

config           coached / total
--------------------------------
filter OFF                1 / 10
filter ON                 6 / 10

Difference: +5


### Step 6: your three sentences  *(write)*

Write what changed and why you think so. A no-improvement result is a real finding: if the number didn't move, say that and give your best guess why. Edit this cell:

> 1. What I changed: …
> 2. What the numbers showed: …
> 3. Why I think that happened: …